## Globalna analiza tplików mp.4 z tempem bicia serca osobników Daphnia magna, w poszczególnych punktach czasowych, w których sprawdzano odpowiednie stężenia

## 1. Konfiguracja i importy

Ta sekcja definiuje ścieżki danych, katalog wyników oraz parametry analizy.

In [ ]:
from pathlib import Path
import re
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import detrend


replicate_name = "20250911"
root_dir = Path("/media/marcelina/marcelina/D. magna") / replicate_name
heart_folder = root_dir
output_dir = Path("/home/marcelina/D.magna/Krótki") / replicate_name
output_dir.mkdir(parents=True, exist_ok=True)

MIN_HEART_HZ = 1.5
MAX_HEART_HZ = 10.0

### Funkcja budowania tabeli nagrań

Funkcja rozpoznaje wyłącznie pliki o oczekiwanej strukturze katalogów i nazw.

In [ ]:
def zbuduj_tabele_nagran(root_dir):
    """Skanuje katalog i buduje tabele opisujaca rozpoznane nagrania."""
    filename_pattern = re.compile(
        r"^(?:(?P<individual>\d+)\s+(?P<measurement>ruch|serce)|"
        r"(?P<measurement_alt>ruch|serce)\s+(?P<individual_alt>\d+))"
        r"(?:_.*)?\.mp4$",
        re.IGNORECASE,
    )
    rows = []
    unmatched_files = []
    video_files = sorted(
        path for path in root_dir.rglob("*.mp4") if path.is_file()
    )

    for video_file in video_files:
        relative_parts = video_file.relative_to(root_dir).parts
        try:
            time_folder, concentration_folder, file_name = relative_parts[-3:]
            time_match = re.fullmatch(r"(\d+)h", time_folder)
            concentration_match = re.fullmatch(
                r"(\d+(?:[.,]\d+)?)\s*mg", concentration_folder
            )
            file_match = filename_pattern.fullmatch(file_name)

            if not time_match or not concentration_match or not file_match:
                unmatched_files.append(str(video_file))
                continue

            individual = file_match.group("individual") or file_match.group("individual_alt")
            measurement = (
                file_match.group("measurement")
                or file_match.group("measurement_alt")
            ).lower()
            rows.append({
                "file_name": file_name,
                "path": str(video_file),
                "data": root_dir.name,
                "punkt_czasowy_h": int(time_match.group(1)),
                "stężenie_mg": float(
                    concentration_match.group(1).replace(",", ".")
                ),
                "osobnik": int(individual),
                "pomiar": measurement,
            })
        except (IndexError, ValueError):
            unmatched_files.append(str(video_file))

    conditions_df = pd.DataFrame(rows).sort_values(
        ["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"],
        na_position="last",
    ).reset_index(drop=True)
    return video_files, conditions_df, rows, unmatched_files


video_files, conditions_df, rows, unmatched_files = zbuduj_tabele_nagran(root_dir)
print(f"Znaleziono plików MP4: {len(video_files)}")
print(f"Rozpoznano poprawnie: {len(conditions_df)}")
if unmatched_files:
    print("Pliki nierozpoznane:")
    for file_path in unmatched_files:
        print(file_path)
display(conditions_df.head())

### Uzupełnienie brakujących nagrań

Brakujące kombinacje osobnik–pomiar są oznaczane jako rekordy bez pliku.

In [ ]:
# Uzupełnienie brakujących nagrań jako zmarłe osobniki
expected_measurements = ("ruch", "serce")
expected_individuals = range(1, 5)

# Wszystkie poprawne kombinacje punkt czasowy–stężenie znalezione w katalogach
expected_conditions = set()

for time_dir in sorted(root_dir.iterdir()):
    if not time_dir.is_dir():
        continue

    time_match = re.fullmatch(r"(\d+)h", time_dir.name)
    if not time_match:
        continue

    for concentration_dir in time_dir.iterdir():
        if not concentration_dir.is_dir():
            continue

        concentration_match = re.fullmatch(
            r"(\d+(?:[.,]\d+)?)\s*mg",
            concentration_dir.name,
        )
        if not concentration_match:
            continue

        expected_conditions.add((
            int(time_match.group(1)),
            int(float(concentration_match.group(1).replace(",", "."))),
        ))

# Klucze nagrań, które już istnieją
existing_keys = {
    (
        row["punkt_czasowy_h"],
        row["stężenie_mg"],
        row["osobnik"],
        row["pomiar"],
    )
    for row in rows
    if row["stężenie_mg"] is not None
}

dead_rows = []

for time_point, concentration in sorted(expected_conditions):
    for individual in expected_individuals:
        for measurement in expected_measurements:
            key = (time_point, concentration, individual, measurement)

            if key not in existing_keys:
                dead_rows.append({
                    "file_name": None,
                    "path": None,
                    "data": root_dir.name,
                    "punkt_czasowy_h": time_point,
                    "stężenie_mg": concentration,
                    "osobnik": individual,
                    "pomiar": measurement,
                })

required_columns = [
    "file_name",
    "path",
    "data",
    "punkt_czasowy_h",
    "stężenie_mg",
    "osobnik",
    "pomiar",
]

dead_df = pd.DataFrame(dead_rows, columns=required_columns)

conditions_df = (
    pd.concat([conditions_df, dead_df], ignore_index=True)
    .sort_values(
        ["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"],
        na_position="last",
    )
    .reset_index(drop=True)
)

print(f"Dodano rekordów zmarłych dafni: {len(dead_df)}")
display(dead_df)

## 3. Kontrola metadanych wideo

Dla każdego rozpoznanego pliku sprawdzane są FPS, rozdzielczość, liczba klatek i czas trwania.

### Funkcja odczytu metadanych

Definicja funkcji używanej w następnej komórce kontrolnej.

In [ ]:
def odczytaj_metadane_wideo(video_path):
    """Odczytuje podstawowe parametry pliku wideo."""
    capture = cv2.VideoCapture(str(video_path))
    metadata = {
        "file_name": Path(video_path).name,
        "path": str(video_path),
        "odczytano": False,
        "fps": np.nan,
        "liczba_klatek": np.nan,
        "szerokosc": np.nan,
        "wysokosc": np.nan,
        "czas_s": np.nan,
        "blad_odczytu": "",
    }

    if not capture.isOpened():
        metadata["blad_odczytu"] = "Nie mozna otworzyc pliku"
        capture.release()
        return metadata

    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
    metadata.update({
        "odczytano": fps > 0 and frame_count > 0,
        "fps": fps,
        "liczba_klatek": int(frame_count),
        "szerokosc": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "wysokosc": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "czas_s": frame_count / fps if fps > 0 else np.nan,
    })
    if not metadata["odczytano"]:
        metadata["blad_odczytu"] = "Nieprawidlowe metadane pliku"
    capture.release()
    return metadata

In [ ]:
# Kontrola parametrów wszystkich nagrań
metadata_rows = [
    odczytaj_metadane_wideo(path)
    for path in conditions_df.loc[conditions_df["path"].notna(), "path"].drop_duplicates()
]
video_metadata_df = pd.DataFrame(metadata_rows)
readable = video_metadata_df["odczytano"].eq(True)

if readable.any():
    reference_width = video_metadata_df.loc[readable, "szerokosc"].mode().iloc[0]
    reference_height = video_metadata_df.loc[readable, "wysokosc"].mode().iloc[0]
    reference_fps = video_metadata_df.loc[readable, "fps"].median()
    reference_duration = video_metadata_df.loc[readable, "czas_s"].median()

    video_metadata_df["blad"] = ""
    video_metadata_df.loc[~readable, "blad"] = "Nie można odczytać pliku; "
    video_metadata_df.loc[
        readable
        & (
            (video_metadata_df["szerokosc"] != reference_width)
            | (video_metadata_df["wysokosc"] != reference_height)
        ),
        "blad",
    ] += "Nieprawidłowa rozdzielczość; "
    video_metadata_df.loc[
        readable & (video_metadata_df["fps"] - reference_fps).abs().gt(1.0),
        "blad",
    ] += "Nietypowe FPS; "
    video_metadata_df.loc[
        readable & (video_metadata_df["czas_s"] - reference_duration).abs().gt(5.0),
        "blad",
    ] += "Nietypowy czas trwania; "

    print(f"Rozdzielczość referencyjna: {reference_width} x {reference_height} px")
    print(f"Mediana FPS: {reference_fps:.2f}")

metadata_errors = video_metadata_df[video_metadata_df["blad"].ne("")]
print(f"Sprawdzono nagrań: {len(video_metadata_df)}")
print(f"Nagrań z uwagami: {len(metadata_errors)}")
display(metadata_errors)

### Funkcje wczytywania nagrań

Funkcje poniżej tylko definiują sposób odczytu. Konkretne pliki są uruchamiane w kolejnych komórkach.

In [ ]:
def znajdz_pliki_serce(folder_path):
    """Zwraca posortowane pliki MP4, których nazwa zawiera slowo 'serce'."""
    folder_path = Path(folder_path)
    return sorted(
        path
        for path in folder_path.rglob("*.mp4")
        if path.is_file() and "serce" in path.stem.lower()
    )


def wczytaj_koncowe_20_s(video_path, max_seconds=20, grayscale=False):
    """Zwraca metadane i generator koncowych sekund nagrania."""
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        raise OSError(f"Nie mozna otworzyc pliku: {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0 or frame_count <= 0:
        capture.release()
        raise ValueError(f"Nieprawidlowe parametry pliku: {video_path}")

    frames_to_keep = min(frame_count, int(round(max_seconds * fps)))
    first_frame = max(0, frame_count - frames_to_keep)
    metadata = {
        "path": str(video_path),
        "fps": fps,
        "liczba_klatek": frames_to_keep,
        "szerokosc": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "wysokosc": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "czas_s": frames_to_keep / fps,
        "poczatkowa_klatka": first_frame,
    }

    def frame_generator():
        try:
            capture.set(cv2.CAP_PROP_POS_FRAMES, first_frame)
            for _ in range(frames_to_keep):
                success, frame = capture.read()
                if not success:
                    break
                if grayscale:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                yield frame
        finally:
            capture.release()

    return metadata, frame_generator()


def wzmacnij_krawedzie_czarne_tlo(frame):
    """Zwraca pelny obraz szarosci z lokalnie wzmocnionymi krawedziami."""
    if frame is None:
        return None
    if frame.ndim == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blurred = cv2.GaussianBlur(frame, (5, 5), 0)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    contrast = clahe.apply(blurred)
    sobel_x = cv2.Sobel(contrast, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(contrast, cv2.CV_32F, 0, 1, ksize=3)
    edges = cv2.magnitude(sobel_x, sobel_y)
    edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    enhanced = cv2.addWeighted(contrast, 1.0, edges, 0.9, 0)
    return cv2.medianBlur(enhanced, 3)


def pobierz_reprezentatywna_klatke(video_path):
    """Wczytuje klatke ze srodka nagrania."""
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise OSError(f"Nie mozna otworzyc pliku: {video_path}")

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_count // 2))
    success, frame = capture.read()
    capture.release()

    if not success or frame is None:
        raise ValueError(f"Nie mozna odczytac klatki: {video_path}")
    return frame

### 4. Wybór ROI na obrazie ze wzmocnionymi krawędziami
Na pełnej reprezentatywnej klatce w skali szarości wzmacniane są krawędzie, a następnie interaktywnie wybierany jest obszar ROI.

#### Funkcje przetwarzania klatek
Definicje obejmują wybór końcowych 20 sekund, konwersję do szarości i wzmocnienie krawędzi.

#### Funkcje analizy obrazu
Przygotowanie pełnej klatki do wyboru ROI i późniejszego pomiaru jasności.

### Komórka wykonawcza: wszystkie pliki serca z powtórzenia 20250909

Komórka działa na wszystkich plikach MP4 zawierających `serce` w całym katalogu `20250909`, czyli dla punktów czasowych 0, 2, 4, 6, 8, 10 i 12 h oraz wszystkich stężeń. ROI zaznaczane jest na pełnym obrazie szarym ze wzmocnionymi krawędziami.

In [ ]:
# Wybór ROI dla wszystkich wskazanych nagrań serca
roi_rows = []
heart_video_files = znajdz_pliki_serce(heart_folder)

if not heart_video_files:
    raise FileNotFoundError(f"Nie znaleziono plików serca w: {heart_folder}")

for video_path in heart_video_files:
    try:
        frame = pobierz_reprezentatywna_klatke(video_path)
    except (OSError, ValueError) as error:
        print(error)
        continue

    enhanced_frame = wzmacnij_krawedzie_czarne_tlo(frame)
    window_name = f"ROI | {video_path.name}"
    print(f"Wybierz ROI na pełnym obrazie szarym ze wzmocnionymi krawędziami: {video_path}")
    print("ENTER/SPACE zatwierdza, ESC pomija nagranie.")
    x, y, width, height = cv2.selectROI(
        window_name,
        enhanced_frame,
        showCrosshair=True,
        fromCenter=False,
    )
    cv2.destroyAllWindows()

    if width == 0 or height == 0:
        print("Pominięto nagranie.")
        continue

    roi_rows.append({
        "file_name": video_path.name,
        "path": str(video_path),
        "roi_x": int(x),
        "roi_y": int(y),
        "roi_width": int(width),
        "roi_height": int(height),
    })

roi_df = pd.DataFrame(roi_rows)
roi_csv_path = output_dir / "roi_coordinates_heart.csv"
roi_df.to_csv(roi_csv_path, index=False, encoding="utf-8-sig")
print(f"Zapisano {len(roi_df)} obszarów ROI: {roi_csv_path}")
display(roi_df)

In [ ]:
# Wybór ROI dla wszystkich wskazanych nagrań serca
roi_rows = []
heart_video_files = znajdz_pliki_serce(heart_folder)

if not heart_video_files:
    raise FileNotFoundError(f"Nie znaleziono plików serca w: {heart_folder}")

for video_path in heart_video_files:
    try:
        frame = pobierz_reprezentatywna_klatke(video_path)
    except (OSError, ValueError) as error:
        print(error)
        continue

    enhanced_frame = wzmacnij_krawedzie_czarne_tlo(frame)
    window_name = f"ROI | {video_path.name}"
    print(f"Wybierz ROI na pełnym obrazie szarym ze wzmocnionymi krawędziami: {video_path}")
    print("ENTER/SPACE zatwierdza, ESC pomija nagranie.")
    x, y, width, height = cv2.selectROI(
        window_name,
        enhanced_frame,
        showCrosshair=True,
        fromCenter=False,
    )
    cv2.destroyAllWindows()

    if width == 0 or height == 0:
        print("Pominięto nagranie.")
        continue

    roi_rows.append({
        "file_name": video_path.name,
        "path": str(video_path),
        "roi_x": int(x),
        "roi_y": int(y),
        "roi_width": int(width),
        "roi_height": int(height),
    })

roi_df = pd.DataFrame(roi_rows)
roi_csv_path = output_dir / "roi_coordinates_heart.csv"
roi_df.to_csv(roi_csv_path, index=False, encoding="utf-8-sig")
print(f"Zapisano {len(roi_df)} obszarów ROI: {roi_csv_path}")
display(roi_df)

## 5. Sygnał jasności ROI

Dla każdej klatki obliczana jest średnia i mediana jasności, a wynik zapisywany jest do CSV.

In [ ]:
# Obliczenie średniej i mediany jasności ROI dla każdej klatki
signal_rows = []
signal_errors = []

video_metadata = (
    conditions_df[
        conditions_df["path"].notna()
        & conditions_df["pomiar"].eq("serce")
    ]
    .drop_duplicates("path")
    .set_index("path")
)

for _, roi_row in roi_df.iterrows():
    video_path = roi_row["path"]
    try:
        video_info, frames = wczytaj_koncowe_20_s(
            video_path,
            max_seconds=20,
            grayscale=True,
        )
        video_row = video_metadata.loc[video_path]
        x = int(roi_row["roi_x"])
        y = int(roi_row["roi_y"])
        width = int(roi_row["roi_width"])
        height = int(roi_row["roi_height"])

        for frame_number, frame in enumerate(frames):
            frame_height, frame_width = frame.shape[:2]
            roi = frame[
                y:min(y + height, frame_height),
                x:min(x + width, frame_width),
            ]
            if roi.size == 0:
                continue

            signal_rows.append({
                "file_name": roi_row["file_name"],
                "path": video_path,
                "data": video_row["data"],
                "punkt_czasowy_h": video_row["punkt_czasowy_h"],
                "stężenie_mg": video_row["stężenie_mg"],
                "osobnik": video_row["osobnik"],
                "pomiar": video_row["pomiar"],
                "klatka": frame_number,
                "czas_s": frame_number / video_info["fps"],
                "jasność_średnia": float(np.mean(roi)),
                "jasność_mediana": float(np.median(roi)),
            })
    except (KeyError, OSError, ValueError, cv2.error) as error:
        signal_errors.append({"path": video_path, "błąd": str(error)})

roi_signal_df = pd.DataFrame(signal_rows)
output_path = output_dir / "roi_signal_df.csv"
roi_signal_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Liczba pomiarów klatkowych: {len(roi_signal_df)}")
print(f"Zapisano wyniki ROI: {output_path}")
display(roi_signal_df.head())
if signal_errors:
    display(pd.DataFrame(signal_errors))

## 6. Przebiegi czasowe i FFT

Przebiegi jasności są grupowane per nagranie, a następnie analizowane pod kątem częstotliwości bicia serca i SNR.

In [ ]:
# Zbudowanie tabeli z jednym przebiegiem czasowym dla każdego nagrania
brightness_timecourses_df = (
    roi_signal_df
    .sort_values(["path", "klatka"])
    .groupby(
        [
            "file_name",
            "path",
            "data",
            "punkt_czasowy_h",
            "stężenie_mg",
            "osobnik",
            "pomiar",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        czas_s=("czas_s", list),
        przebieg_czasowy_jasności=("jasność_średnia", list),
        przebieg_mediany_jasności=("jasność_mediana", list),
    )
)

brightness_timecourses_df.to_pickle(output_dir / "brightness_timecourses.pkl")
print(f"Zapisano przebiegi: {output_dir / 'brightness_timecourses.pkl'}")
display(brightness_timecourses_df.head())

In [ ]:
# Wykres przebiegu czasowego i widma FFT dla jednego nagrania

from scipy.signal import detrend
example_signal = brightness_timecourses_df.iloc[0]

czas = np.asarray(example_signal["czas_s"], dtype=float)
sygnal = np.asarray(
    example_signal["przebieg_czasowy_jasności"],
    dtype=float,
)

# Usunięcie trendu i okno Hanna
sygnal_clean = detrend(sygnal, type="linear")
sygnal_windowed = sygnal_clean * np.hanning(len(sygnal_clean))

# Transformata Fouriera
dt = np.median(np.diff(czas))
fps = 1 / dt

widmo = np.fft.rfft(sygnal_windowed)
czestotliwosci = np.fft.rfftfreq(len(sygnal_windowed), d=dt)
amplituda = np.abs(widmo) / len(sygnal_windowed)

# Zakres częstotliwości bicia serca
maska = (
    (czestotliwosci >= MIN_HEART_HZ)
    & (czestotliwosci <= MAX_HEART_HZ)
)

peak_index = np.flatnonzero(maska)[
    np.argmax(amplituda[maska])
]
peak_frequency = czestotliwosci[peak_index]
bpm = peak_frequency * 60

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(czas, sygnal, label="Jasność ROI")
axes[0].plot(czas, sygnal_windowed, label="Po detrendowaniu i oknie Hanna")
axes[0].set_title(
    f"Przebieg czasowy: {example_signal['file_name']}"
)
axes[0].set_xlabel("Czas [s]")
axes[0].set_ylabel("Jasność")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(czestotliwosci, amplituda)
axes[1].axvline(
    peak_frequency,
    color="red",
    linestyle="--",
    label=f"{peak_frequency:.2f} Hz ({bpm:.1f} BPM)",
)
axes[1].set_xlim(0, MAX_HEART_HZ)
axes[1].set_title("Widmo amplitudowe FFT")
axes[1].set_xlabel("Częstotliwość [Hz]")
axes[1].set_ylabel("Amplituda")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Częstotliwość próbkowania: {fps:.2f} Hz")
print(f"Dominująca częstotliwość: {peak_frequency:.3f} Hz")
print(f"Tempo bicia serca: {bpm:.1f} BPM")

## 7. Analiza częstotliwości i SNR

Funkcja poniżej oblicza dominującą częstotliwość, BPM oraz SNR. Następna komórka uruchamia ją dla wszystkich przebiegów.

In [ ]:
def analizuj_przebieg_serca(czas, sygnal):
    """Wyznacza dominujaca czestotliwosc oraz SNR z przebiegu jasnosci."""
    czas = np.asarray(czas, dtype=float)
    sygnal = np.asarray(sygnal, dtype=float)
    valid = np.isfinite(czas) & np.isfinite(sygnal)
    czas = czas[valid]
    sygnal = sygnal[valid]

    if len(sygnal) < 16:
        raise ValueError("Za malo probek do analizy FFT.")

    sort_order = np.argsort(czas)
    czas = czas[sort_order]
    sygnal = sygnal[sort_order]
    dt = np.median(np.diff(czas))
    if dt <= 0:
        raise ValueError("Nieprawidlowa os czasu.")

    fps = 1 / dt
    czas_uniform = np.arange(czas[0], czas[-1], dt)
    sygnal_uniform = np.interp(czas_uniform, czas, sygnal)
    sygnal_clean = detrend(sygnal_uniform, type="linear")
    sygnal_windowed = sygnal_clean * np.hanning(len(sygnal_clean))

    widmo = np.fft.rfft(sygnal_windowed)
    czestotliwosci = np.fft.rfftfreq(len(sygnal_windowed), d=dt)
    moc = np.abs(widmo) ** 2
    maska = (
        (czestotliwosci >= MIN_HEART_HZ)
        & (czestotliwosci <= min(MAX_HEART_HZ, fps / 2))
    )
    if not np.any(maska):
        raise ValueError("Brak czestotliwosci w zadanym zakresie.")

    indeksy = np.flatnonzero(maska)
    peak_index = indeksy[np.argmax(moc[maska])]
    peak_frequency = czestotliwosci[peak_index]

    if 0 < peak_index < len(moc) - 1:
        y_left, y_peak, y_right = np.log(
            np.maximum(moc[peak_index - 1:peak_index + 2], 1e-30)
        )
        denominator = y_left - 2 * y_peak + y_right
        if denominator != 0:
            correction = 0.5 * (y_left - y_right) / denominator
            peak_frequency += correction * (czestotliwosci[1] - czestotliwosci[0])

    noise_mask = maska.copy()
    exclusion_bins = 2
    low = max(0, peak_index - exclusion_bins)
    high = min(len(noise_mask), peak_index + exclusion_bins + 1)
    noise_mask[low:high] = False
    noise_power = np.median(moc[noise_mask])
    peak_power = moc[peak_index]
    snr_db = 10 * np.log10(peak_power / noise_power) if noise_power > 0 else np.inf

    return {
        "fps": fps,
        "ft_heart_hz": peak_frequency,
        "bpm": peak_frequency * 60,
        "snr_db": snr_db,
        "peak_power": peak_power,
        "noise_power": noise_power,
        "liczba_próbek": len(sygnal_uniform),
    }

Dla każdego osobnika i każdego stężenia wygeneruj wykres częstotliwości od godizny. Dla każdego osobnika i każdego stężenia wygeneruj wykres częstotliwości od godizny 0h do 24h, pokazujący zmiany częstotliwości bicia serca w czasie. Wykres powinien mieć oś X reprezentującą czas w godzinach, a oś Y reprezentującą częstotliwość bicia serca w Hz. Każdy osobnik powinien być reprezentowany innym kolorem lub stylem linii, a legenda powinna wskazywać, który kolor lub styl odpowiada któremu osobnikowi. Ososbnik dla każdego stężenia są podpisane tak samo osobnik 1 serce w stężeniu 10 mg nie jest tym samym co 1 serce w stężeniu 40 mg

In [ ]:
# Wykresy częstotliwości bicia serca od czasu dla każdego stężenia

if heart_rate_results_df.empty:
    print("Brak wyników FFT do przedstawienia.")
else:
    plot_df = (
        heart_rate_results_df
        .dropna(
            subset=[
                "punkt_czasowy_h",
                "stężenie_mg",
                "osobnik",
                "ft_heart_hz",
            ]
        )
        .groupby(
            ["stężenie_mg", "osobnik", "punkt_czasowy_h"],
            as_index=False,
        )["ft_heart_hz"]
        .mean()
    )

    for concentration, concentration_df in plot_df.groupby(
        "stężenie_mg",
        sort=True,
    ):
        fig, ax = plt.subplots(figsize=(10, 6))

        individuals = sorted(concentration_df["osobnik"].unique())
        colors = plt.cm.tab10(
            np.linspace(0, 1, max(len(individuals), 1))
        )

        for color, individual in zip(colors, individuals):
            individual_df = (
                concentration_df[
                    concentration_df["osobnik"] == individual
                ]
                .sort_values("punkt_czasowy_h")
            )

            ax.plot(
                individual_df["punkt_czasowy_h"],
                individual_df["ft_heart_hz"],
                marker="o",
                linewidth=2,
                color=color,
                label=f"Osobnik {int(individual)}",
            )

        ax.set_xlim(0, 24)
        ax.set_xticks(np.arange(0, 25, 4))
        ax.set_xlabel("Czas [h]")
        ax.set_ylabel("Częstotliwość bicia serca [Hz]")
        ax.set_title(
            f"Zmiana częstotliwości bicia serca w czasie\n"
            f"Stężenie: {concentration:g} mg"
        )
        ax.grid(True, alpha=0.3)
        ax.legend(title="Osobnik")
        fig.tight_layout()

        output_path = (
            output_dir / f"heart_frequency_{concentration:g}mg.png"
        )
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print(f"Zapisano wykres: {output_path}")

#### Segmentacja

In [ ]:
# Eksperyment: segmentacja ruchu serca i pomiar częstotliwości skurczów
# Metoda wykorzystuje różnicę między kolejnymi klatkami w istniejącym ROI.
# Pole maski ruchu tworzy sygnał czasowy, który następnie analizujemy FFT.

from scipy.signal import find_peaks


def zmierz_sygnał_segmentacji(video_path, roi_row, max_seconds=20):
    """Segmentuje zmiany ruchu w ROI i zwraca sygnał pola maski."""
    video_info, frames = wczytaj_koncowe_20_s(
        video_path,
        max_seconds=max_seconds,
        grayscale=True,
    )
    x = int(roi_row["roi_x"])
    y = int(roi_row["roi_y"])
    width = int(roi_row["roi_width"])
    height = int(roi_row["roi_height"])
    kernel = np.ones((3, 3), dtype=np.uint8)
    previous_roi = None
    signal = []
    diagnostic_frame = None
    diagnostic_mask = None

    for frame in frames:
        frame_height, frame_width = frame.shape[:2]
        roi = frame[
            y:min(y + height, frame_height),
            x:min(x + width, frame_width),
        ]
        if roi.size == 0:
            continue

        roi = cv2.GaussianBlur(roi, (5, 5), 0)
        if previous_roi is None:
            previous_roi = roi
            continue

        difference = cv2.absdiff(roi, previous_roi)
        threshold_value, mask = cv2.threshold(
            difference,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU,
        )
        threshold_value = max(threshold_value, 3.0)
        mask = cv2.threshold(difference, threshold_value, 255, cv2.THRESH_BINARY)[1]
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        # Usuwamy drobny szum i zostawiamy dominujący obszar ruchu.
        component_count, labels, statistics, _ = cv2.connectedComponentsWithStats(
            mask,
            connectivity=8,
        )
        if component_count > 1:
            largest_component = 1 + np.argmax(statistics[1:, cv2.CC_STAT_AREA])
            mask = np.where(labels == largest_component, 255, 0).astype(np.uint8)

        signal.append(float(np.count_nonzero(mask) / mask.size))
        if diagnostic_frame is None:
            diagnostic_frame = roi.copy()
            diagnostic_mask = mask.copy()
        previous_roi = roi

    if len(signal) < 16:
        raise ValueError("Za mało klatek po segmentacji do analizy FFT.")

    return video_info, np.asarray(signal), diagnostic_frame, diagnostic_mask


segmentation_results = []
segmentation_errors = []
first_segmentation_diagnostic = None

for _, roi_row in roi_df.iterrows():
    try:
        video_path = roi_row["path"]
        video_info, segmentation_signal, diagnostic_frame, diagnostic_mask = (
            zmierz_sygnał_segmentacji(video_path, roi_row)
        )
        times = np.arange(len(segmentation_signal)) / video_info["fps"]
        frequency_result = analizuj_przebieg_serca(times, segmentation_signal)
        video_row = video_metadata.loc[video_path]
        segmentation_results.append({
            "file_name": roi_row["file_name"],
            "path": video_path,
            "data": video_row["data"],
            "punkt_czasowy_h": video_row["punkt_czasowy_h"],
            "stężenie_mg": video_row["stężenie_mg"],
            "osobnik": video_row["osobnik"],
            "pomiar": video_row["pomiar"],
            "metoda": "segmentacja ruchu ROI",
            **frequency_result,
            "średnie_pole_maski": float(np.mean(segmentation_signal)),
            "odchylenie_pola_maski": float(np.std(segmentation_signal)),
        })
        if first_segmentation_diagnostic is None:
            first_segmentation_diagnostic = (
                roi_row["file_name"],
                times,
                segmentation_signal,
                diagnostic_frame,
                diagnostic_mask,
            )
    except (KeyError, OSError, ValueError, cv2.error) as error:
        segmentation_errors.append({
            "path": roi_row["path"],
            "błąd": str(error),
        })

segmentation_results_df = (
    pd.DataFrame(segmentation_results)
    .sort_values(["punkt_czasowy_h", "stężenie_mg", "osobnik"])
    .reset_index(drop=True)
)

print(f"Przeanalizowano segmentacją: {len(segmentation_results_df)} nagrań")
display(segmentation_results_df)
if segmentation_errors:
    print("Błędy segmentacji:")
    display(pd.DataFrame(segmentation_errors))

if first_segmentation_diagnostic is not None:
    file_name, diagnostic_times, diagnostic_signal, diagnostic_frame, diagnostic_mask = (
        first_segmentation_diagnostic
    )
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].imshow(diagnostic_frame, cmap="gray")
    axes[0].set_title("ROI")
    axes[1].imshow(diagnostic_mask, cmap="gray")
    axes[1].set_title("Maska ruchu")
    axes[2].plot(diagnostic_times, diagnostic_signal)
    axes[2].set_title("Pole maski w czasie")
    axes[2].set_xlabel("Czas [s]")
    axes[2].set_ylabel("Udział pikseli maski")
    for axis in axes[:2]:
        axis.axis("off")
    fig.suptitle(f"Diagnostyka segmentacji: {file_name}")
    fig.tight_layout()
    segmentation_diagnostic_path = output_dir / "segmentation_diagnostic.png"
    fig.savefig(segmentation_diagnostic_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Zapisano diagnostykę segmentacji: {segmentation_diagnostic_path}")

# Porównanie częstotliwości z metodą jasności ROI, jeśli oba wyniki są dostępne.
if not segmentation_results_df.empty and not heart_rate_results_df.empty:
    comparison_df = segmentation_results_df.merge(
        heart_rate_results_df[
            ["path", "ft_heart_hz", "bpm"]
        ],
        on="path",
        suffixes=("_segmentacja", "_jasność"),
    )
    comparison_df["różnica_hz"] = (
        comparison_df["ft_heart_hz_segmentacja"]
        - comparison_df["ft_heart_hz_jasność"]
    ).abs()
    display(
        comparison_df[
            [
                "file_name",
                "ft_heart_hz_segmentacja",
                "ft_heart_hz_jasność",
                "różnica_hz",
                "snr_db",
            ]
        ]
    )

In [ ]:
# Diagnostyka każdego nagrania: jasność i segmentacja w czasie oraz w FFT.
# Kliknij peaki na panelach FFT. Enter zatwierdza nagranie, Esc pomija.

import matplotlib

matplotlib.use("TkAgg", force=True)
import matplotlib.pyplot as plt
from scipy.signal import detrend


def przygotuj_fft(czas, sygnal):
    czas = np.asarray(czas, dtype=float)
    sygnal = np.asarray(sygnal, dtype=float)
    valid = np.isfinite(czas) & np.isfinite(sygnal)
    czas = czas[valid]
    sygnal = sygnal[valid]
    if len(sygnal) < 16:
        raise ValueError("Za malo probek do analizy FFT.")

    sort_order = np.argsort(czas)
    czas = czas[sort_order]
    sygnal = sygnal[sort_order]
    dt = np.median(np.diff(czas))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Nieprawidlowa os czasu.")

    czas_uniform = np.arange(czas[0], czas[-1] + dt / 2, dt)
    sygnal_uniform = np.interp(czas_uniform, czas, sygnal)
    sygnal_clean = detrend(sygnal_uniform, type="linear")
    sygnal_windowed = sygnal_clean * np.hanning(len(sygnal_clean))
    frequencies = np.fft.rfftfreq(len(sygnal_windowed), d=dt)
    amplitudes = np.abs(np.fft.rfft(sygnal_windowed)) / len(sygnal_windowed)
    return czas, sygnal, frequencies, amplitudes


def ustaw_okno_matplotlib(fig, title):
    """Otwiera figure w osobnym, aktywnym oknie jak okno wyboru ROI."""
    manager = fig.canvas.manager
    try:
        manager.set_window_title(title)
    except AttributeError:
        pass
    try:
        manager.window.deiconify()
        manager.window.lift()
        manager.window.focus_force()
    except AttributeError:
        pass


def pobierz_info_nagrania(video_path):
    """Zwraca metadane także dla plików z sufiksem, np. _trim_20s."""
    if video_path in video_metadata.index:
        return video_metadata.loc[video_path]

    path = Path(video_path)
    if len(path.parts) < 3:
        raise KeyError(f"Brak metadanych nagrania: {video_path}")

    time_match = re.fullmatch(r"(\d+)h", path.parent.parent.name)
    concentration_match = re.fullmatch(
        r"(\d+(?:[.,]\d+)?)\s*mg",
        path.parent.name,
    )
    file_match = re.match(
        r"^(?P<individual>\d+)\s+(?P<measurement>ruch|serce)(?:_.*)?\.mp4$",
        path.name,
        flags=re.IGNORECASE,
    )
    if not time_match or not concentration_match or not file_match:
        raise KeyError(f"Nie mozna rozpoznac metadanych nagrania: {video_path}")

    return {
        "data": root_dir.name,
        "punkt_czasowy_h": int(time_match.group(1)),
        "stężenie_mg": float(concentration_match.group(1).replace(",", ".")),
        "osobnik": int(file_match.group("individual")),
        "pomiar": file_match.group("measurement").lower(),
    }


def wybierz_peaki(fig, axes, file_name, recording_info):
    selected_peaks = []
    finished = {"value": False}

    def on_click(event):
        if event.inaxes not in (axes[1], axes[3]) or event.xdata is None:
            return

        method = "jasnosc" if event.inaxes is axes[1] else "segmentacja"
        frequency = float(event.xdata)
        selected_peaks.append({
            **recording_info,
            "file_name": file_name,
            "metoda": method,
            "peak_hz": frequency,
            "peak_bpm": frequency * 60,
        })
        event.inaxes.axvline(
            frequency,
            color="crimson",
            linestyle="--",
            linewidth=1.5,
        )
        fig.canvas.draw_idle()
        print(
            f"{file_name} | {method}: "
            f"{frequency:.4f} Hz = {frequency * 60:.1f} BPM"
        )

    def on_key(event):
        if event.key in ("enter", "return", "escape"):
            finished["value"] = event.key != "escape"
            plt.close(fig)

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    ustaw_okno_matplotlib(fig, f"Diagnostyka | {file_name}")
    plt.show(block=True)
    return selected_peaks, finished["value"]


manual_peak_results = []

for _, roi_row in roi_df.iterrows():
    file_name = roi_row["file_name"]
    video_path = roi_row["path"]

    try:
        metadata_row = pobierz_info_nagrania(video_path)
        recording_info = {
            "path": video_path,
            "data": metadata_row["data"],
            "punkt_czasowy_h": metadata_row["punkt_czasowy_h"],
            "stężenie_mg": metadata_row["stężenie_mg"],
            "osobnik": metadata_row["osobnik"],
        }

        brightness_rows = (
            roi_signal_df[roi_signal_df["path"] == video_path]
            .sort_values("czas_s")
        )
        if brightness_rows.empty:
            raise ValueError("Brak przebiegu jasnosci ROI.")

        brightness_time, brightness_signal, brightness_freq, brightness_fft = przygotuj_fft(
            brightness_rows["czas_s"].to_numpy(dtype=float),
            brightness_rows["jasność_średnia"].to_numpy(dtype=float),
        )

        video_info, segmentation_signal, _, _ = zmierz_sygnał_segmentacji(
            video_path,
            roi_row,
        )
        segmentation_time = np.arange(len(segmentation_signal)) / video_info["fps"]
        segmentation_time, segmentation_signal, segmentation_freq, segmentation_fft = przygotuj_fft(
            segmentation_time,
            segmentation_signal,
        )
    except (KeyError, OSError, ValueError, cv2.error) as error:
        print(f"Pominieto {file_name}: {error}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(16, 9), squeeze=False)
    axes = axes.ravel()
    axes[0].plot(brightness_time, brightness_signal, color="tab:blue")
    axes[0].set_title("Jasnosc ROI w czasie")
    axes[0].set_xlabel("Czas [s]")
    axes[0].set_ylabel("Srednia jasnosc")

    axes[1].plot(brightness_freq, brightness_fft, color="tab:blue")
    axes[1].set_title("FFT jasnosci: kliknij peaki")
    axes[1].set_xlabel("Czestotliwosc [Hz]")
    axes[1].set_ylabel("Amplituda")
    axes[1].set_xlim(0, MAX_HEART_HZ)

    axes[2].plot(segmentation_time, segmentation_signal, color="tab:orange")
    axes[2].set_title("Pole maski segmentacji w czasie")
    axes[2].set_xlabel("Czas [s]")
    axes[2].set_ylabel("Udzial pikseli maski")

    axes[3].plot(segmentation_freq, segmentation_fft, color="tab:orange")
    axes[3].set_title("FFT segmentacji: kliknij peaki")
    axes[3].set_xlabel("Czestotliwosc [Hz]")
    axes[3].set_ylabel("Amplituda")
    axes[3].set_xlim(0, MAX_HEART_HZ)

    for axis in axes:
        axis.grid(True, alpha=0.3)
    fig.suptitle(
        f"{file_name} | Enter: dalej, Esc: pomin | "
        f"{recording_info['stężenie_mg']:g} mg, osobnik {int(recording_info['osobnik'])}"
    )
    fig.tight_layout()

    selected_peaks, accepted = wybierz_peaki(
        fig,
        axes,
        file_name,
        recording_info,
    )
    if accepted:
        manual_peak_results.extend(selected_peaks)

manual_peak_results_df = pd.DataFrame(manual_peak_results)
manual_peak_path = output_dir / "manual_peak_results.csv"
manual_peak_results_df.to_csv(manual_peak_path, index=False, encoding="utf-8-sig")
print(f"Zapisano ręcznie zaznaczone peaki: {manual_peak_path}")
display(manual_peak_results_df)

In [ ]:
# Tabela porównawcza: dwie wartości automatyczne i dwie ręcznie zaznaczone

manual_by_path = (
    manual_peak_results_df
    .sort_values("metoda")
    .drop_duplicates(["path", "metoda"], keep="last")
    .pivot(index="path", columns="metoda", values="peak_hz")
    .rename(columns={
        "jasnosc": "reczna_jasnosc_hz",
        "segmentacja": "reczna_segmentacja_hz",
    })
    .reset_index()
)

auto_brightness = heart_rate_results_df[
    ["path", "ft_heart_hz"]
].rename(columns={"ft_heart_hz": "auto_jasnosc_hz"})

auto_segmentation = segmentation_results_df[
    ["path", "ft_heart_hz"]
].rename(columns={"ft_heart_hz": "auto_segmentacja_hz"})

recording_metadata = (
    conditions_df[
        conditions_df["path"].notna()
        & conditions_df["pomiar"].eq("serce")
    ]
    [[
        "path",
        "file_name",
        "data",
        "punkt_czasowy_h",
        "stężenie_mg",
        "osobnik",
        "pomiar",
    ]]
    .drop_duplicates("path")
)

heart_rate_comparison_df = (
    recording_metadata
    .merge(auto_brightness, on="path", how="left")
    .merge(auto_segmentation, on="path", how="left")
    .merge(manual_by_path, on="path", how="left")
    .sort_values(["punkt_czasowy_h", "stężenie_mg", "osobnik"])
    .reset_index(drop=True)
)

comparison_path = output_dir / "heart_rate_comparison.csv"
heart_rate_comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)
print(f"Zapisano tabelę porównawczą: {comparison_path}")
missing_auto = heart_rate_comparison_df[
    heart_rate_comparison_df[["auto_jasnosc_hz", "auto_segmentacja_hz"]]
    .isna()
    .any(axis=1)
]
if not missing_auto.empty:
    print("Nagrania serca bez co najmniej jednego wyniku automatycznego:")
    display(missing_auto[["file_name", "path", "auto_jasnosc_hz", "auto_segmentacja_hz"]])
display(
    heart_rate_comparison_df[
        [
            "file_name",
            "punkt_czasowy_h",
            "stężenie_mg",
            "osobnik",
            "auto_jasnosc_hz",
            "auto_segmentacja_hz",
            "reczna_jasnosc_hz",
            "reczna_segmentacja_hz",
        ]
    ]
)

## 8. Indeks jednego powtórzenia

Analiza obejmuje wyłącznie powtórzenie `20250909`, ale skanuje wszystkie jego punkty czasowe, stężenia oraz pliki `serce.mp4`. Identyfikator osobnika pozostaje stały w obrębie tego powtórzenia i stężenia, więc osobnik 1 przy 0 mg jest śledzony między punktami czasowymi, ale nie jest łączony z osobnikiem 1 przy innym stężeniu.

In [ ]:
# Indeks wszystkich nagrań serca w jednym powtórzeniu i checkpoint CSV
if not root_dir.is_dir():
    raise FileNotFoundError(f"Brak katalogu powtórzenia: {root_dir}")

video_files, indexed_conditions_df, indexed_rows, unmatched_files = zbuduj_tabele_nagran(root_dir)

all_conditions_df = (
    indexed_conditions_df
    .assign(
        powtórzenie=lambda frame: replicate_name,
        osobnik_id=lambda frame: (
            frame["powtórzenie"].astype(str)
            + "_"
            + frame["stężenie_mg"].map(lambda value: f"{value:g}")
            + "mg_"
            + frame["osobnik"].astype("Int64").astype(str)
        ),
    )
    .sort_values(["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"])
    .reset_index(drop=True)
)

all_conditions_path = output_dir / "all_recordings_index.csv"
all_conditions_df.to_csv(all_conditions_path, index=False, encoding="utf-8-sig")

expected_time_points = {0, 2, 4, 6, 8, 10, 12}
expected_concentrations = {0.0, 2.0, 10.0, 20.0, 40.0}
expected_video_count = len(expected_time_points) * len(expected_concentrations) * 8
heart_video_count = int(all_conditions_df["pomiar"].eq("serce").sum())

print(f"Powtórzenie: {replicate_name}")
print(f"Znaleziono plików MP4: {len(video_files)}")
print(f"Oczekiwana liczba wszystkich plików: {expected_video_count}")
print(f"Rozpoznano rekordów: {len(all_conditions_df)}")
print(f"Znaleziono plików serca: {heart_video_count}")
print(f"Oczekiwana liczba plików serca: {expected_video_count // 2}")
print(f"Nierozpoznane pliki: {len(unmatched_files)}")
print(f"Checkpoint indeksu: {all_conditions_path}")

missing_expected = (
    pd.MultiIndex.from_product(
        [sorted(expected_time_points), sorted(expected_concentrations), range(1, 5), ["serce", "ruch"]],
        names=["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"],
    )
    .to_frame(index=False)
    .merge(
        all_conditions_df[["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"]],
        on=["punkt_czasowy_h", "stężenie_mg", "osobnik", "pomiar"],
        how="left",
        indicator=True,
    )
    .loc[lambda frame: frame["_merge"].eq("left_only")]
    .drop(columns="_merge")
)
missing_expected.to_csv(output_dir / "missing_expected_recordings.csv", index=False, encoding="utf-8-sig")

if unmatched_files:
    display(pd.DataFrame({"path": unmatched_files}))
if not missing_expected.empty:
    print(f"Brakujące oczekiwane rekordy: {len(missing_expected)}")
    display(missing_expected)
display(all_conditions_df.head())

In [ ]:
# Ustawienie całego jednego powtórzenia dla dalszej analizy
conditions_df = all_conditions_df.copy()
heart_folder = root_dir
print(f"Ustawiono powtórzenie {replicate_name} dla dalszej analizy.")
print(f"Katalog nagrań serca: {heart_folder}")
print("Po tej komórce uruchom ponownie komórki ROI, sygnału, FFT i statystyk.")

## 9. Statystyki BPM, modele i raport zbiorczy

Analiza korzysta z automatycznej częstotliwości z metody jasności ROI (`heart_rate_results_df`). Wnioski statystyczne są wykonywane na poziomie nagrania; klucz `osobnik_id` zachowuje zależność podłużną osobnika w obrębie powtórzenia i stężenia.

In [ ]:
# Zbiorcza analiza statystyczna BPM
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

analysis_df = heart_rate_results_df.copy()
if analysis_df.empty:
    raise ValueError("heart_rate_results_df jest pusta. Najpierw uruchom analizę FFT.")

analysis_df["bpm"] = analysis_df["ft_heart_hz"] * 60
analysis_df["stężenie_mg"] = analysis_df["stężenie_mg"].astype(float)
analysis_df["powtórzenie"] = analysis_df["data"].astype(str)
analysis_df["osobnik_id"] = (
    analysis_df["powtórzenie"]
    + "_"
    + analysis_df["stężenie_mg"].map(lambda value: f"{value:g}")
    + "mg_"
    + analysis_df["osobnik"].astype("Int64").astype(str)
)
analysis_df = analysis_df.dropna(subset=["bpm", "stężenie_mg", "punkt_czasowy_h"])
analysis_df.to_csv(output_dir / "bpm_analysis_checkpoint.csv", index=False, encoding="utf-8-sig")

summary_by_concentration = (
    analysis_df.groupby("stężenie_mg")["bpm"]
    .agg(
        liczba=("count"),
        średnia_bpm=("mean"),
        odchylenie_std_bpm=("std"),
        mediana_bpm=("median"),
        minimum_bpm=("min"),
        maksimum_bpm=("max"),
    )
    .reset_index()
)
summary_by_concentration.to_csv(
    output_dir / "bpm_summary_by_concentration.csv", index=False, encoding="utf-8-sig"
)

normality_rows = []
for concentration, group in analysis_df.groupby("stężenie_mg"):
    values = group["bpm"].dropna().to_numpy()
    if len(values) >= 3:
        test_values = values if len(values) <= 5000 else np.random.default_rng(42).choice(values, 5000, replace=False)
        statistic, p_value = stats.shapiro(test_values)
        normality_rows.append({
            "stężenie_mg": concentration,
            "n": len(values),
            "shapiro_stat": statistic,
            "p_value": p_value,
            "wniosek": "brak podstaw do odrzucenia normalności" if p_value >= 0.05 else "odrzucono normalność",
        })
normality_df = pd.DataFrame(normality_rows)
normality_df.to_csv(output_dir / "bpm_normality_tests.csv", index=False, encoding="utf-8-sig")

concentration_levels = analysis_df["stężenie_mg"].nunique()
time_levels = analysis_df["punkt_czasowy_h"].nunique()
one_way_groups = [group["bpm"].to_numpy() for _, group in analysis_df.groupby("stężenie_mg")]
if len(one_way_groups) >= 2:
    one_way_stat, one_way_p = stats.f_oneway(*one_way_groups)
else:
    one_way_stat, one_way_p = np.nan, np.nan
one_way_df = pd.DataFrame([{
    "test": "ANOVA jednoczynnikowa",
    "czynnik": "stężenie_mg",
    "F": one_way_stat,
    "p_value": one_way_p,
    "uwaga": "wymagane są co najmniej dwa stężenia" if len(one_way_groups) < 2 else "",
}])
one_way_df.to_csv(output_dir / "anova_one_way.csv", index=False, encoding="utf-8-sig")

if concentration_levels >= 2 and time_levels >= 2:
    two_way_model = ols("bpm ~ C(stężenie_mg) * C(punkt_czasowy_h)", data=analysis_df).fit()
    two_way_df = anova_lm(two_way_model, typ=1).reset_index().rename(columns={"index": "źródło"})
else:
    two_way_df = pd.DataFrame([{
        "źródło": "ANOVA dwuczynnikowa",
        "uwaga": "wymagane są co najmniej dwa stężenia i dwa punkty czasowe",
    }])
two_way_df.to_csv(output_dir / "anova_two_way.csv", index=False, encoding="utf-8-sig")

if len(one_way_groups) >= 2:
    tukey_result = pairwise_tukeyhsd(
        endog=analysis_df["bpm"],
        groups=analysis_df["stężenie_mg"].map(lambda value: f"{value:g} mg"),
        alpha=0.05,
    )
    tukey_df = pd.DataFrame(
        tukey_result._results_table.data[1:],
        columns=tukey_result._results_table.data[0],
    )
else:
    tukey_df = pd.DataFrame({"uwaga": ["Tukey HSD wymaga co najmniej dwóch stężeń."]})
tukey_df.to_csv(output_dir / "tukey_hsd_concentration.csv", index=False, encoding="utf-8-sig")

concentration_means = summary_by_concentration[["stężenie_mg", "średnia_bpm"]].dropna()
model_rows = []
if len(concentration_means) >= 3:
    linear_model = ols("średnia_bpm ~ stężenie_mg", data=concentration_means).fit()
    quadratic_model = ols("średnia_bpm ~ stężenie_mg + I(stężenie_mg ** 2)", data=concentration_means).fit()
    selected_model = quadratic_model if quadratic_model.aic < linear_model.aic else linear_model
    model_kind = "kwadratowa" if selected_model is quadratic_model else "liniowa"
    for name, model in (("liniowa", linear_model), ("kwadratowa", quadratic_model)):
        model_rows.append({
            "model": name,
            "aic": model.aic,
            "r_squared": model.rsquared,
            "p_value_model": model.f_pvalue,
            "model_wybrany": name == model_kind,
        })
    with open(output_dir / "bpm_regression_report.txt", "w", encoding="utf-8") as report_file:
        report_file.write(selected_model.summary().as_text())
else:
    model_rows.append({"model": "brak", "uwaga": "Regresja wymaga co najmniej trzech stężeń."})
model_summary_df = pd.DataFrame(model_rows)
model_summary_df.to_csv(output_dir / "bpm_regression_models.csv", index=False, encoding="utf-8-sig")

import seaborn as sns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.boxplot(data=analysis_df, x="stężenie_mg", y="bpm", ax=axes[0, 0], color="skyblue")
axes[0, 0].set(title="BPM względem stężenia", xlabel="Stężenie [mg]", ylabel="BPM")
sns.scatterplot(data=analysis_df, x="stężenie_mg", y="bpm", hue="powtórzenie", style="osobnik", ax=axes[0, 1])
axes[0, 1].set(title="BPM względem stężenia", xlabel="Stężenie [mg]", ylabel="BPM")
line_df = analysis_df.groupby(["stężenie_mg", "punkt_czasowy_h"], as_index=False)["bpm"].mean()
sns.lineplot(data=line_df, x="punkt_czasowy_h", y="bpm", hue="stężenie_mg", marker="o", palette="viridis", ax=axes[1, 0])
axes[1, 0].set(title="Średnie BPM względem czasu", xlabel="Czas [h]", ylabel="Średnie BPM")
heatmap_df = analysis_df.pivot_table(index="stężenie_mg", columns="punkt_czasowy_h", values="bpm", aggfunc="mean")
sns.heatmap(heatmap_df.sort_index(), annot=True, fmt=".1f", cmap="magma", ax=axes[1, 1])
axes[1, 1].set(title="Heatmapa średniego BPM", xlabel="Czas [h]", ylabel="Stężenie [mg]")
fig.tight_layout()
fig.savefig(output_dir / "bpm_summary_plots.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Checkpoint analizy: {output_dir / 'bpm_analysis_checkpoint.csv'}")
print("Statystyki podstawowe:")
display(summary_by_concentration)
print("Testy normalności:")
display(normality_df)
print("ANOVA jednoczynnikowa:")
display(one_way_df)
print("ANOVA dwuczynnikowa:")
display(two_way_df)
print("Tukey HSD:")
display(tukey_df)
print("Modele regresji:")
display(model_summary_df)

### Wizualizacja wyników każdej analizy

Każdy test statystyczny otrzymuje osobny wykres diagnostyczny, aby wynik liczbowy można było ocenić razem z rozkładem i wielkością efektu.

In [ ]:
# Osobne wizualizacje dla rozkładów, ANOVA, Tukeya i regresji
import math
import seaborn as sns
from statsmodels.graphics.gofplots import qqplot

sns.set_theme(style="whitegrid")

# 1. Rozkłady i wykresy Q-Q dla każdego stężenia.
concentrations = sorted(analysis_df["stężenie_mg"].dropna().unique())
fig, axes = plt.subplots(len(concentrations), 2, figsize=(14, max(4, 3.5 * len(concentrations))), squeeze=False)
for row_index, concentration in enumerate(concentrations):
    values = analysis_df.loc[analysis_df["stężenie_mg"] == concentration, "bpm"].dropna()
    sns.histplot(values, kde=len(values) >= 2, ax=axes[row_index, 0], color="steelblue")
    axes[row_index, 0].set_title(f"Rozkład BPM: {concentration:g} mg")
    axes[row_index, 0].set_xlabel("BPM")
    axes[row_index, 0].set_ylabel("Liczba obserwacji")
    if len(values) >= 3:
        qqplot(values, line="45", ax=axes[row_index, 1])
        axes[row_index, 1].set_title(f"Q-Q plot: {concentration:g} mg")
    else:
        axes[row_index, 1].text(0.5, 0.5, "Za mało obserwacji do Q-Q plot", ha="center", va="center")
        axes[row_index, 1].set_axis_off()
fig.tight_layout()
fig.savefig(output_dir / "01_rozkłady_i_qq_bpm.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 2. ANOVA jednoczynnikowa: średnie, odchylenia standardowe i wynik testu.
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    summary_by_concentration["stężenie_mg"],
    summary_by_concentration["średnia_bpm"],
    yerr=summary_by_concentration["odchylenie_std_bpm"].fillna(0),
    fmt="o-",
    capsize=5,
    linewidth=2,
    color="tab:blue",
)
ax.set_title(f"ANOVA jednoczynnikowa: F={one_way_stat:.3g}, p={one_way_p:.3g}")
ax.set_xlabel("Stężenie [mg]")
ax.set_ylabel("Średnie BPM ± SD")
fig.tight_layout()
fig.savefig(output_dir / "02_anova_jednoczynnikowa.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 3. ANOVA dwuczynnikowa: wykres interakcji stężenie × czas.
fig, ax = plt.subplots(figsize=(11, 6))
if concentration_levels >= 2 and time_levels >= 2:
    interaction_df = analysis_df.groupby(
        ["punkt_czasowy_h", "stężenie_mg"], as_index=False
    )["bpm"].mean()
    sns.lineplot(
        data=interaction_df,
        x="punkt_czasowy_h",
        y="bpm",
        hue="stężenie_mg",
        marker="o",
        palette="viridis",
        ax=ax,
    )
    ax.set_title("ANOVA dwuczynnikowa: wykres interakcji")
    ax.set_xlabel("Punkt czasowy [h]")
    ax.set_ylabel("Średnie BPM")
else:
    ax.text(0.5, 0.5, "Za mało poziomów do wykresu interakcji", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(output_dir / "03_anova_dwuczynnikowa_interakcja.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 4. Tukey HSD: różnice średnich i 95% przedziały ufności.
fig, ax = plt.subplots(figsize=(11, max(5, 0.45 * len(tukey_df))))
if {"group1", "group2", "meandiff", "lower", "upper"}.issubset(tukey_df.columns):
    tukey_plot = tukey_df.copy()
    tukey_plot["porównanie"] = tukey_plot["group1"] + " vs " + tukey_plot["group2"]
    tukey_plot = tukey_plot.sort_values("meandiff")
    y_positions = np.arange(len(tukey_plot))
    ax.errorbar(
        tukey_plot["meandiff"],
        y_positions,
        xerr=[
            tukey_plot["meandiff"] - tukey_plot["lower"],
            tukey_plot["upper"] - tukey_plot["meandiff"],
        ],
        fmt="o",
        capsize=4,
        color="tab:purple",
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(tukey_plot["porównanie"])
    ax.set_xlabel("Różnica średnich BPM z 95% CI")
    ax.set_title("Tukey HSD: porównania par stężeń")
else:
    ax.text(0.5, 0.5, "Tukey HSD niedostępny dla aktualnej liczby grup", ha="center", va="center")
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(output_dir / "04_tukey_hsd.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 5. Regresja: średnie BPM, dopasowanie liniowe i kwadratowe.
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    concentration_means["stężenie_mg"],
    concentration_means["średnia_bpm"],
    s=80,
    color="black",
    label="Średnie obserwowane",
)
if len(concentration_means) >= 3:
    x_grid = np.linspace(concentration_means["stężenie_mg"].min(), concentration_means["stężenie_mg"].max(), 200)
    prediction_frame = pd.DataFrame({"stężenie_mg": x_grid})
    ax.plot(x_grid, linear_model.predict(prediction_frame), label="Model liniowy")
    ax.plot(x_grid, quadratic_model.predict(prediction_frame), label="Model kwadratowy")
    ax.set_title(f"Regresja BPM względem stężenia; wybrano: {model_kind}")
else:
    ax.set_title("Regresja: za mało poziomów stężenia")
ax.set_xlabel("Stężenie [mg]")
ax.set_ylabel("Średnie BPM")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "05_regresja_bpm.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# 6. Zbiorczy raport wizualny: czas, stężenie i powtórzenia.
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.lineplot(data=analysis_df, x="punkt_czasowy_h", y="bpm", hue="stężenie_mg", estimator="mean", errorbar="sd", marker="o", ax=axes[0])
sns.boxplot(data=analysis_df, x="stężenie_mg", y="bpm", ax=axes[1], color="lightgreen")
sns.scatterplot(data=analysis_df, x="punkt_czasowy_h", y="bpm", hue="powtórzenie", style="stężenie_mg", ax=axes[2])
axes[0].set_title("Tempo vs czas")
axes[1].set_title("Tempo vs stężenie")
axes[2].set_title("Powtórzenia w czasie")
for axis in axes:
    axis.set_ylabel("BPM")
fig.tight_layout()
fig.savefig(output_dir / "06_raport_zbiorczy_bpm.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Zapisano wizualizacje analiz: rozkłady, ANOVA, Tukey, regresję i raport zbiorczy.")

### Analiza repeated measures dla tempa bicia serca
Sekcja rozszerza poprzednią analizę (ANOVA jednoczynnikowa, dwuczynnikowa "naiwna" i regresja) o właściwy model dla danych z powtarzanymi pomiarami: ten sam osobnik (osobnik_id) mierzony wielokrotnie w kolejnych punktach czasowych. Kolejność: (a) testy założeń, (b) model główny (dwuczynnikowa ANOVA z powtarzanymi pomiarami / mixed ANOVA) z post-hoc, (c) model preferowany (LMM) oraz alternatywa nieparametryczna.

In [ ]:
import pingouin as pg
import scikit_posthocs as sp
import statsmodels.formula.api as smf
from scipy import stats as scipy_stats

# --- Shapiro-Wilk na resztach modelu (nie na surowych wartosciach w grupach) ---
assumptions_model = smf.ols(
    "bpm ~ C(stężenie_mg) * C(punkt_czasowy_h)", data=analysis_df
).fit()
residuals = assumptions_model.resid
shapiro_stat, shapiro_p = scipy_stats.shapiro(residuals)
normality_ok = shapiro_p >= 0.05
print(f"Shapiro-Wilk (reszty modelu): W={shapiro_stat:.4f}, p={shapiro_p:.4g}")
print("Normalność reszt:", "OK" if normality_ok else "ZŁAMANA")

# --- Levene: homogenicznosc wariancji miedzy grupami stezenie x czas ---
analysis_df["_grupa_lev"] = (
    analysis_df["stężenie_mg"].astype(str) + "_" + analysis_df["punkt_czasowy_h"].astype(str)
)
levene_groups = [g["bpm"].to_numpy() for _, g in analysis_df.groupby("_grupa_lev")]
levene_stat, levene_p = scipy_stats.levene(*levene_groups, center="median")
homogeneity_ok = levene_p >= 0.05
print(f"\nLevene: stat={levene_stat:.4f}, p={levene_p:.4g}")
print("Homogeniczność wariancji:", "OK" if homogeneity_ok else "ZŁAMANA")

# --- Mauchly: sferycznosc (wymaga zbalansowanych danych: ten sam osobnik_id we wszystkich punktach czasowych) ---
_time_levels_count = analysis_df["punkt_czasowy_h"].nunique()
_balanced_ids = (
    analysis_df.groupby("osobnik_id")["punkt_czasowy_h"].nunique() == _time_levels_count
)
balanced_ids_heart = _balanced_ids[_balanced_ids].index
balanced_df_heart = analysis_df[analysis_df["osobnik_id"].isin(balanced_ids_heart)]

sphericity_ok = None
if balanced_df_heart["osobnik_id"].nunique() >= 3:
    try:
        spher_result = pg.sphericity(
            data=balanced_df_heart, dv="bpm", within="punkt_czasowy_h", subject="osobnik_id"
        )
        sphericity_ok = bool(spher_result.spher)
        print(f"\nMauchly (sferyczność): W={spher_result.W:.4g}, chi2={spher_result.chi2:.4g}, "
              f"dof={spher_result.dof}, p={spher_result.pval:.4g}")
        print("Sferyczność:", "zachowana" if sphericity_ok else "złamana (wymagana korekta GG/HF)")
    except Exception as error:
        print("Nie udało się policzyć sferyczności:", error)
else:
    print("Za mało zbalansowanych osobników (ten sam osobnik we wszystkich punktach czasowych) do testu sferyczności.")

assumptions_summary_heart = pd.DataFrame([{
    "test": "Shapiro-Wilk (reszty)", "statystyka": shapiro_stat, "p_value": shapiro_p,
    "wniosek": "normalność OK" if normality_ok else "normalność złamana",
}, {
    "test": "Levene", "statystyka": levene_stat, "p_value": levene_p,
    "wniosek": "homogeniczność OK" if homogeneity_ok else "homogeniczność złamana",
}])
assumptions_summary_heart.to_csv(output_dir / "heart_assumptions_summary.csv", index=False, encoding="utf-8-sig")


In [ ]:
mixed_anova_heart = pg.mixed_anova(
    data=analysis_df, dv="bpm", within="punkt_czasowy_h",
    between="stężenie_mg", subject="osobnik_id",
)
mixed_anova_heart.to_csv(output_dir / "heart_mixed_anova.csv", index=False, encoding="utf-8-sig")
print("Mixed ANOVA (split-plot): stężenie [między] x czas [wewnątrz]")
if sphericity_ok is False:
    print("Sferyczność złamana -> odczytaj kolumnę 'p-GG-corr' (korekta Greenhouse-Geissera) dla efektów within/interaction.")
display(mixed_anova_heart)

# --- Post-hoc: Tukey HSD dla efektu głównego stężenia ---
tukey_heart = pg.pairwise_tukey(data=analysis_df, dv="bpm", between="stężenie_mg")
tukey_heart.to_csv(output_dir / "heart_tukey_hsd.csv", index=False, encoding="utf-8-sig")
print("\nTukey HSD (porównania między wszystkimi stężeniami, po odsumowaniu czasu):")
display(tukey_heart)

# --- Post-hoc: Dunnett (każde stężenie vs kontrola) osobno dla każdego punktu czasowego ---
_control_value = float(analysis_df["stężenie_mg"].min())  # zaklada, ze najnizsze stezenie = kontrola (0 mg)
dunnett_rows_heart = []
for time_point, group in analysis_df.groupby("punkt_czasowy_h"):
    control_sample = group.loc[group["stężenie_mg"] == _control_value, "bpm"].to_numpy()
    treatment_concentrations = sorted(c for c in group["stężenie_mg"].unique() if c != _control_value)
    treatment_samples = [group.loc[group["stężenie_mg"] == c, "bpm"].to_numpy() for c in treatment_concentrations]
    if len(control_sample) == 0 or any(len(s) == 0 for s in treatment_samples):
        continue
    dunnett_result = scipy_stats.dunnett(*treatment_samples, control=control_sample, random_state=42)
    for concentration, p_value, stat_value in zip(treatment_concentrations, dunnett_result.pvalue, dunnett_result.statistic):
        dunnett_rows_heart.append({
            "punkt_czasowy_h": time_point, "stężenie_mg": concentration,
            "statystyka": stat_value, "p_value": p_value,
        })
dunnett_df_heart = pd.DataFrame(dunnett_rows_heart)
dunnett_df_heart.to_csv(output_dir / "heart_dunnett_vs_control.csv", index=False, encoding="utf-8-sig")
print("\nDunnett (każde stężenie vs kontrola, osobno dla każdego punktu czasowego):")
display(dunnett_df_heart)

In [ ]:
lmm_heart = smf.mixedlm(
    "bpm ~ stężenie_mg * punkt_czasowy_h",
    data=analysis_df,
    groups=analysis_df["osobnik_id"],
    vc_formula={"powtorzenie": "0 + C(powtórzenie)"},
)
lmm_heart_fit = lmm_heart.fit(reml=True)
print("LMM: bpm ~ stężenie * czas + (1|osobnik_id) + (1|powtórzenie)")
print(lmm_heart_fit.summary())

with open(output_dir / "heart_lmm_report.txt", "w", encoding="utf-8") as report_file:
    report_file.write(lmm_heart_fit.summary().as_text())
]

In [ ]:
kw_rows_heart = []
dunn_frames_heart = []
for time_point, group in analysis_df.groupby("punkt_czasowy_h"):
    samples = [g["bpm"].to_numpy() for _, g in group.groupby("stężenie_mg")]
    if len(samples) >= 2:
        kw_stat, kw_p = scipy_stats.kruskal(*samples)
        kw_rows_heart.append({"punkt_czasowy_h": time_point, "H": kw_stat, "p_value": kw_p})
        dunn_result = sp.posthoc_dunn(group, val_col="bpm", group_col="stężenie_mg", p_adjust="holm")
        dunn_result["punkt_czasowy_h"] = time_point
        dunn_frames_heart.append(dunn_result)

kw_df_heart = pd.DataFrame(kw_rows_heart)
kw_df_heart.to_csv(output_dir / "heart_kruskal_wallis.csv", index=False, encoding="utf-8-sig")
print("Kruskal-Wallis (osobno dla każdego punktu czasowego, porównanie stężeń):")
display(kw_df_heart)

if dunn_frames_heart:
    dunn_all_heart = pd.concat(dunn_frames_heart)
    dunn_all_heart.to_csv(output_dir / "heart_dunn_posthoc.csv", encoding="utf-8-sig")
    print("\nDunn post-hoc (macierze p skorygowane metodą Holma, osobno dla każdego czasu) zapisano do CSV.")

friedman_rows_heart = []
for concentration, group in balanced_df_heart.groupby("stężenie_mg"):
    pivoted = group.pivot_table(index="osobnik_id", columns="punkt_czasowy_h", values="bpm").dropna()
    if pivoted.shape[0] >= 3 and pivoted.shape[1] >= 3:
        friedman_stat, friedman_p = scipy_stats.friedmanchisquare(*[pivoted[c].to_numpy() for c in pivoted.columns])
        friedman_rows_heart.append({
            "stężenie_mg": concentration, "n_osobników": pivoted.shape[0],
            "chi2": friedman_stat, "p_value": friedman_p,
        })
friedman_df_heart = pd.DataFrame(friedman_rows_heart)
friedman_df_heart.to_csv(output_dir / "heart_friedman.csv", index=False, encoding="utf-8-sig")
print("\nFriedman (przebieg czasowy w obrębie każdego stężenia):")
display(friedman_df_heart)


In [ ]:
# Srednie BPM wg stezenia i czasu - srednia z wynikow (heart_rate_comparison.csv) z 3 folderow
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_OUTPUT_DIR = Path("/home/marcelina/D.magna/Krótki")
REPLICATE_NAMES = ["20250909", "20250910", "20250911"]

frames = []
for rep_name in REPLICATE_NAMES:
    rep_path = BASE_OUTPUT_DIR / rep_name / "heart_rate_comparison.csv"
    if not rep_path.exists():
        raise FileNotFoundError(f"Brak pliku dla powtórzenia {rep_name}: {rep_path}")
    frames.append(pd.read_csv(rep_path))

combined_df = pd.concat(frames, ignore_index=True)

# BPM z automatycznej metody jasnosci ROI (kolumna auto_jasnosc_hz -> BPM)
combined_df["bpm"] = combined_df["auto_jasnosc_hz"] * 60
combined_df = combined_df.dropna(subset=["bpm", "stężenie_mg", "punkt_czasowy_h"])

print(f"Polaczono {len(REPLICATE_NAMES)} powtorzen, lacznie {len(combined_df)} nagran.")

# Srednie BPM wg stezenia (po zsumowaniu czasu)
srednie_bpm_stezenie = combined_df.groupby("stężenie_mg")["bpm"].agg(["mean", "std", "count"]).round(2)
print("\nSrednie BPM wg stezenia:")
display(srednie_bpm_stezenie)

# Srednie BPM wg stezenia i czasu
srednie_bpm_czas = (
    combined_df.groupby(["stężenie_mg", "punkt_czasowy_h"])["bpm"]
    .mean()
    .unstack()
    .round(1)
)
print("\nSrednie BPM wg stezenia i czasu:")
display(srednie_bpm_czas)

# Wykres: srednie BPM w czasie, dla kazdego stezenia
fig, ax = plt.subplots(figsize=(9, 6))
for concentration, row in srednie_bpm_czas.iterrows():
    ax.plot(row.index, row.values, marker="o", label=f"{concentration:g} mg")
ax.set_xlabel("Czas [h]")
ax.set_ylabel("Średnie BPM (z 3 powtórzeń)")
ax.set_title("Średnie BPM w czasie, dla każdego stężenia (3 powtórzenia)")
ax.legend(title="Stężenie")
plt.tight_layout()

fig.savefig(BASE_OUTPUT_DIR / "srednie_bpm_3_powtorzenia.png", dpi=300, bbox_inches="tight")
plt.show()